In [ ]:

# Supp F1 A, F, Supp F5 C, D, E

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime
import math
from matplotlib.pyplot import cm
import statsmodels.api as sm
from paths import DATA_DIR, fig_dir

In [ ]:
#custom schema
from alison_rlmodel import BehaviorModelResults # changes wd for jl code!


In [ ]:
from plot_content import *
from find_my_data import *
from plot_rlmodel import *
from plot_figs_summary_metrics import get_all_rat_stable_nwb_file_names

### figure and param setup and load data

In [ ]:
from fig_helpers import *

set_figure_defaults()

fig_path = fig_dir('figs26')

In [ ]:

save_fig = False

In [ ]:
# Params
behavior_model_params_name = 'beta_stable_withleaf' #'default_hmm_0623'    #'hmm_test' #'default_hmm'

position_info_param_name='default_decoding'
remove_hpd_timepoints = True
hpd_percent = 50
hpd_threshold = 50
require_nonlocal_by_segment = False
remove_low_speed_timepoints = True
head_speed_threshold = 10

In [ ]:
# load data
out_path = f'{DATA_DIR}/big_df_pkls/'
# today_now = datetime.now().strftime("%Y%m%d") 
today_now = '20240212'
subject_ids = ['senor', 'chimi', 'j16', 'wilbur', 'peanut']

big_dfs = {}
for subject_id in subject_ids:
    try:
        big_dfs[subject_id] = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
    except Exception as e:
        print('exception',e)

In [ ]:
stable_nwbs = {}
clusterless_nwbs = {}
stable_clusterless_nwbs = {}
for subject_id in subject_ids:
    stable_nwbs[subject_id] = list( (Session & {'session_description LIKE "Spatial bandit task (regular)"'}
                                             & {"subject_id": subject_id}).fetch('nwb_file_name') )
    clusterless_nwbs[subject_id] = list(np.unique((ClusterlessAcausalResultsSummary()
                                                   & spatial_bandit_query_by_rat(rat_list=[subject_id])).fetch('nwb_file_name')))
    if subject_id == 'j16':
        stable_nwbs['j16'].remove('mediumnwb20230802_.nwb')
    if subject_id == 'chimi':
        stable_nwbs['chimi'].remove('chimi20200216_new_.nwb')
    if subject_id == 'senor':
        stable_nwbs['senor'].remove('senor20201030_.nwb')

    stable_clusterless_nwbs[subject_id] = [nwb for nwb in clusterless_nwbs[subject_id] if nwb in stable_nwbs[subject_id]]

print(stable_clusterless_nwbs)

is_mapped_seg_a_leaf_map = {0:False, 1:True, 2:True, 3:False, 4:True, 5:True, 6:False, 7:True, 8:True}
segs_to_patch_map = {0:1, 1:1, 2:1, 3:2, 4:2, 5:2, 6:3, 7:3, 8:3}

# get to stable data only
all_rat_big_dfs_stable = {}
for subject_id in subject_ids:
    df = big_dfs[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_clusterless_nwbs[subject_id])]
    df_stable['is_actual_seg_mapped_a_leaf'] = df_stable[['actual_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['is_mental_seg_mapped_a_leaf'] = df_stable[['mental_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['mental_patch_mapped'] = df_stable[['mental_segment_mapped']].applymap(segs_to_patch_map.get)
    all_rat_big_dfs_stable[subject_id] = df_stable
    
for subject_id in subject_ids:
    df = all_rat_big_dfs_stable[subject_id]
    p_rew_cols = [f"p_rew_leaf{i}" for i in [1,2,3,4,5,6]]
    all_rat_big_dfs_stable[subject_id] = all_rat_big_dfs_stable[subject_id][~all_rat_big_dfs_stable[subject_id][p_rew_cols].eq(all_rat_big_dfs_stable[subject_id]['p_rew_leaf1'], axis=0).all(axis=1)]

In [ ]:
hmm_results = {}

for subject_id in subject_ids:
    hmm_result = (BehaviorModelResults() & {'behavior_model_params_name': behavior_model_params_name, 'subject_id': subject_id}).fetch1_dataframe()
    hmm_results[subject_id] = hmm_result

hmm_results_stable = {}
for subject_id in subject_ids:
    df = hmm_results[subject_id]
    stable_nwbs = stable_clusterless_nwbs[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_nwbs)]
    hmm_results_stable[subject_id] = df_stable

for subject_id in subject_ids:
    df = hmm_results_stable[subject_id]
    #p_rew_cols = [f"p_rew_leaf{i}" for i in [1,2,3,4,5,6]]
    hmm_results_stable[subject_id] = df[np.logical_and(df['contingency']!=100100100100100100, df['contingency']!=505050505050)]

In [ ]:
hmm_results_stable['all_rats'] = pd.concat([hmm_results_stable[subject_id] for subject_id in subject_ids])

In [ ]:
n_total=0
for subject_id in subject_ids:
    print(f'{subject_id}, n= {len(hmm_results_stable[subject_id])} trials')
    print(f'   SWITCH: {len(hmm_results_stable[subject_id][hmm_results_stable[subject_id].stemswitch==1])}')
    print(f'   Stay:   {len(hmm_results_stable[subject_id][hmm_results_stable[subject_id].stemswitch==0])}')
    n_total += len(hmm_results_stable[subject_id])
print(n_total)

In [ ]:
data_new_dict = {}

for subject_id in subject_ids:
    
    data_df = hmm_results_stable[subject_id]

    # Identify stem switch trials, and handle first trial from the shift
    data_df['stem_id'] = data_df['stem'].replace({'A':1, 'B':2, 'C':3})
    data_df['stem_shifted'] = data_df.groupby(by = ['nwb_file_name','epoch'])['stem'].shift(1).bfill(limit=1)
    data_df['stem_switch'] = data_df['stem'] != data_df['stem_shifted']

    # Calculate trials from prior/next switch
    data_df['trials_from_prior_switch_groups'] = data_df.groupby(by = ['nwb_file_name','epoch'])['stem_switch'].cumsum()
    data_df['trials_from_prior_switch'] = data_df.groupby(by = ['nwb_file_name','epoch','trials_from_prior_switch_groups']).cumcount()
    data_df['trials_from_prior_switch'] = data_df['trials_from_prior_switch'].where(data_df['trials_from_prior_switch_groups'].gt(0), np.nan)

#     # This is the og line, dysfunctions with one group setup
#     # data_df['trials_to_next_switch_groups'] = data_df.groupby(by = ['nwb_file_name','epoch']).apply(lambda x_df: x_df['stem_switch'].shift(fill_value=False).cumsum()).reset_index(name='group').set_index('id')['group']

#     # New version
#     tmp = data_df.groupby(by = ['nwb_file_name','epoch']).apply(lambda x_df: x_df['stem_switch'].shift(fill_value=False
#                 ).cumsum()).reset_index().melt(id_vars=['nwb_file_name','epoch'], var_name='id', value_name='group').set_index('id')
#     data_df['trials_to_next_switch_groups'] = tmp['group']

#     data_df['trials_to_next_switch'] = data_df.groupby(by=['nwb_file_name','epoch','trials_to_next_switch_groups'])['trials_to_next_switch_groups'].cumcount(ascending=False)

#     # Ensure that don't count trials to end of epoch as trials to next switch 
#     data_df = data_df.groupby(by=['nwb_file_name','epoch']).apply(_set_nan_at_max)    
#     data_df['trials_from_next_switch'] = data_df['trials_to_next_switch_with_nans']*-1

    # Calculate bout lengths
    data_df['try_bout_idx'] = data_df.groupby(by = ['nwb_file_name','epoch'])['stem'].transform(lambda x_df: (x_df != x_df.shift(1)).cumsum())
    data_df['bout_len'] = data_df.groupby(by = ['nwb_file_name','epoch', 'try_bout_idx'])['try_bout_idx'].transform(len)
    data_df['bout_len_new'] = data_df.groupby(by = ['nwb_file_name','epoch', 'try_bout_idx'])['trials_from_prior_switch'].transform(lambda x: np.max(x))
    
    data_new_dict[subject_id] = data_df

# Make all animal concatenated df
data_new_dict_concatenated_df = pd.concat(data_new_dict.values(), keys=data_new_dict.keys(), names=['subject_id'])

# Reset the index to make subject_id a column
data_new_dict_concatenated_df.reset_index(level=0, inplace=True)
data_new_dict_concatenated_df.reset_index(drop=False, inplace=True)
data_new_dict_concatenated_df

In [ ]:
### SF1a

In [ ]:
# make boxplot version
set_figure_defaults()

def plot_bout_len_by_rat_violin(concatenated_df, subject_ids, figwidth, figheight, fig_path, density_norm,
                                              rat_colors=iter(cm.tab20b([0,.8, .85, .1, .05])),inner='quart',
                                                  save_fig = False,
                                                 ):
    fig, ax = plt.subplots(figsize=(figwidth,figheight))  # Adjust the figure size as needed
    new_df = concatenated_df.groupby(by=['subject_id','nwb_file_name','epoch','try_bout_idx',]).apply(lambda x_df: pd.Series({'bout_len_new2': x_df['bout_len_new'].mean(),})).reset_index()
    sns.violinplot(x='subject_id', y='bout_len_new2', data=new_df, palette=rat_colors,
                cut=0,density_norm=density_norm,inner=inner, fill=False)
    medians = new_df.groupby('subject_id')['bout_len_new2'].median().values
    means = new_df.groupby('subject_id')['bout_len_new2'].mean().values
    for i, median in enumerate(medians):
        plt.text(i+.3, 60, f'{median:.2f}', horizontalalignment='center', size='medium', color='black')
    for i, mean in enumerate(means):
        plt.text(i+.3, 70, f'{mean:.2f}', horizontalalignment='center', size='medium', color='black')
    plt.xlabel('Subject', y=-.15)
    plt.xticks([0,1,2,3,4],[f'Rat {subject_id[0].upper()}' for subject_id in subject_ids])
    plt.ylabel('Trials in Patch Before Switch')
    #plt.ylim(0,53)
    sns.despine()
    fig_name = f'all_rat_bout_len_new_violin_densitynorm{density_norm}_inner{inner}_w{figwidth}_h{figheight}'
    if save_fig:
        save_figure(fig_path,fig_name)
    plt.show()

In [ ]:
# save_fig = False

custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))

density_norm="area" #'area' 'width''count'
figwidth = 6
figheight = figwidth*GOLDEN_RATIO
inner='quart'
plot_bout_len_by_rat_violin(data_new_dict_concatenated_df, subject_ids,
                            figwidth=figwidth,
                            figheight=figheight,
                            fig_path=fig_path,
                            density_norm=density_norm,
                            rat_colors=custom_colors_by_rat, 
                            inner=inner,
                            save_fig = save_fig,
                            )